# OOP 6: Polymorphism, Duck Typing, and Protocols

## Table of Contents

1. [Polymorphism](#1)
2. [Mixins: Composing Behaviour Without Deep Hierarchies](#2)
3. [isinstance() and Type Checking](#3)
4. [Duck Typing](#4)
5. [The Tension: Type Hints Meet Duck Typing](#5)
6. [Protocols: Structural Typing Without Inheritance](#6)
7. [Exercise: A Saveable Model](#7)

---
## 1. Polymorphism <a id="1"></a>

**Polymorphism** means code that operates on an interface works correctly for any implementation of that interface — present or future — without modification.

You already saw this in notebook 04: `ExperimentRunner` calls `loader.load()` without caring whether `loader` is a `CSVLoader`, a `DatabaseLoader`, or a test mock. That's polymorphism.

Here's the same idea made explicit with the preprocessor hierarchy:

In [ ]:
import pandas as pd
import numpy as np
from abc import ABC, abstractmethod
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


class BasePreprocessor(ABC):
    @abstractmethod
    def fit(self, X: pd.DataFrame) -> "BasePreprocessor": ...
    @abstractmethod
    def transform(self, X: pd.DataFrame) -> pd.DataFrame: ...
    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)
    def __repr__(self): return f"{type(self).__name__}()"


class StandardScaler(BasePreprocessor):
    def __init__(self):
        self._mean = self._std = None
    def fit(self, X):
        self._mean = X.mean(); self._std = X.std().clip(lower=1e-8); return self
    def transform(self, X):
        return (X - self._mean) / self._std

class MinMaxScaler(BasePreprocessor):
    def __init__(self):
        self._min = self._range = None
    def fit(self, X):
        self._min = X.min(); self._range = (X.max() - X.min()).clip(lower=1e-8); return self
    def transform(self, X):
        return (X - self._min) / self._range

class RobustScaler(BasePreprocessor):
    def __init__(self):
        self._median = self._iqr = None
    def fit(self, X):
        self._median = X.median()
        self._iqr = (X.quantile(0.75) - X.quantile(0.25)).clip(lower=1e-8)
        return self
    def transform(self, X):
        return (X - self._median) / self._iqr

class LogTransformer(BasePreprocessor):
    def fit(self, X): return self
    def transform(self, X): return np.log1p(X)


# This function is written once. It works for all current and future preprocessors.
def evaluate_preprocessors(
    preprocessors: list[BasePreprocessor],
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
) -> pd.DataFrame:
    results = []
    for p in preprocessors:
        X_tr = p.fit_transform(X_train)
        X_te = p.transform(X_test)
        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_tr, y_train)
        acc = accuracy_score(y_test, model.predict(X_te))
        results.append({"preprocessor": repr(p), "accuracy": round(acc, 4)})
    return pd.DataFrame(results).sort_values("accuracy", ascending=False)


np.random.seed(42)
X_all = pd.DataFrame({"tenure":  np.random.randint(1, 60, 300),
                       "charges": np.random.uniform(20, 100, 300)})
y_all = pd.Series((np.random.rand(300) > 0.65).astype(int))
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

print(evaluate_preprocessors(
    [StandardScaler(), MinMaxScaler(), RobustScaler(), LogTransformer()],
    X_tr, X_te, y_tr, y_te
))

The key insight: `evaluate_preprocessors` doesn't have a single `if isinstance(p, StandardScaler)` branch. It just calls `fit_transform()` and `transform()`. Adding a new preprocessor type next week requires zero changes to this function.

This is the practical payoff of ABCs combined with inheritance.

---
## 2. Mixins: Composing Behaviour Without Deep Hierarchies <a id="2"></a>

Sometimes you want to add a capability to several unrelated classes — without creating a deep inheritance tree or duplicating code.

**Mixins** are small classes designed to be "mixed in" as an extra parent. Each mixin does exactly one thing. The naming convention is to end with `Mixin`.

Consider: you want *some* preprocessors to be saveable, and *some* models to be saveable, but `BasePreprocessor` and `BaseModel` are unrelated. You could write `save()`/`load()` in both hierarchies — or write it once in a mixin.

In [ ]:
class PersistenceMixin:
    """Adds save() and load() to any class. Does one thing."""

    def save(self, path: str) -> None:
        import pickle
        with open(path, "wb") as f:
            pickle.dump(self, f)
        print(f"Saved {type(self).__name__} to {path}")

    @classmethod
    def load(cls, path: str):
        import pickle
        with open(path, "rb") as f:
            return pickle.load(f)


class FittedCheckMixin:
    """Adds _require_fitted() to any class with a _fitted attribute."""

    def _require_fitted(self) -> None:
        if not getattr(self, "_fitted", False):
            raise RuntimeError(
                f"{type(self).__name__} is not fitted. Call fit() first."
            )


# Mix both into a StandardScaler — multiply inherit from left to right
class PersistableScaler(FittedCheckMixin, PersistenceMixin, BasePreprocessor):
    def __init__(self):
        self._fitted = False
        self._mean = self._std = None

    def fit(self, X: pd.DataFrame) -> "PersistableScaler":
        self._mean = X.mean()
        self._std = X.std().clip(lower=1e-8)
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        self._require_fitted()   # from FittedCheckMixin
        return (X - self._mean) / self._std


import tempfile, os

X = pd.DataFrame({"tenure": [12.0, 5.0, 24.0], "charges": [70.0, 30.0, 90.0]})
scaler = PersistableScaler()

# Guard works before fitting
try:
    scaler.transform(X)
except RuntimeError as e:
    print(f"RuntimeError: {e}")

# Fit, save, reload, use
scaler.fit(X)
with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as f:
    tmp = f.name
try:
    scaler.save(tmp)
    reloaded = PersistableScaler.load(tmp)
    print(reloaded.transform(X).round(2))
finally:
    os.unlink(tmp)

### MRO — which class wins?

With multiple parents, Python uses the **C3 MRO** to decide which method runs. Inspect it:

In [ ]:
print("PersistableScaler MRO:")
for cls in PersistableScaler.__mro__:
    print(f"  {cls.__name__}")

The rule of thumb: list mixins **before** the base class. `FittedCheckMixin` and `PersistenceMixin` appear before `BasePreprocessor`, so their methods take precedence if there's a name clash.

---
## 3. isinstance() and Type Checking <a id="3"></a>

`isinstance(obj, SomeClass)` checks the full inheritance chain. Always prefer it over `type(x) == SomeClass`, which breaks with subclasses.

In [ ]:
objects = [StandardScaler(), MinMaxScaler(), RobustScaler(), LogTransformer(), PersistableScaler()]

for obj in objects:
    print(
        f"{type(obj).__name__:22s}"
        f"  BasePreprocessor: {isinstance(obj, BasePreprocessor)}"
        f"  PersistenceMixin: {isinstance(obj, PersistenceMixin)}"
        f"  FittedCheckMixin: {isinstance(obj, FittedCheckMixin)}"
    )

---
## 4. Exercise: A Saveable Model <a id="4"></a>

Create a `BaseModel` ABC with `fit(X, y)` and `predict(X)`.

Then implement `PersistableRandomForest` that:
1. Inherits from `PersistenceMixin`, `FittedCheckMixin`, and `BaseModel`
2. Wraps `sklearn`'s `RandomForestClassifier`
3. Calls `_require_fitted()` inside `predict()` before forwarding to the sklearn model
4. Can be saved and reloaded with `save()` / `load()`

In [ ]:
from sklearn.ensemble import RandomForestClassifier

class BaseModel(ABC):
    @abstractmethod
    def fit(self, X: pd.DataFrame, y: pd.Series) -> "BaseModel": ...
    @abstractmethod
    def predict(self, X: pd.DataFrame) -> np.ndarray: ...


class PersistableRandomForest(PersistenceMixin, FittedCheckMixin, BaseModel):
    pass   # TODO


# Test
np.random.seed(0)
X_all = pd.DataFrame({"a": np.random.randn(100), "b": np.random.randn(100)})
y_all = pd.Series((X_all["a"] > 0).astype(int))
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=0)

model = PersistableRandomForest()

# Should raise before fit
try:
    model.predict(X_te)
except RuntimeError as e:
    print(f"RuntimeError: {e}")

# Fit, save, reload, predict
model.fit(X_tr, y_tr)
with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as f:
    tmp = f.name
try:
    model.save(tmp)
    reloaded = PersistableRandomForest.load(tmp)
    acc = accuracy_score(y_te, reloaded.predict(X_te))
    print(f"Accuracy after reload: {acc:.3f}")
finally:
    os.unlink(tmp)

> <details>
> <summary>Solution</summary>
>
> ```python
> class PersistableRandomForest(PersistenceMixin, FittedCheckMixin, BaseModel):
>     def __init__(self, n_estimators: int = 100, random_state: int = 42):
>         self._fitted = False
>         self._model = RandomForestClassifier(
>             n_estimators=n_estimators, random_state=random_state
>         )
>
>     def fit(self, X, y):
>         self._model.fit(X, y)
>         self._fitted = True
>         return self
>
>     def predict(self, X):
>         self._require_fitted()
>         return self._model.predict(X)
> ```
> </details>

---
## 4. Duck Typing <a id="4"></a>

> *"If it walks like a duck and quacks like a duck, it's a duck."*

**Duck typing** means Python doesn't check what *type* an object is — it checks whether it has the methods you're about to call. If it does, the call works. If it doesn't, you get an `AttributeError` at runtime.

This is fundamentally different from languages like Java, where you must declare types upfront and the compiler enforces them.

Sklearn is built entirely on duck typing. `cross_val_score` doesn't check that your estimator inherits from `BaseEstimator` — it checks that it has `.fit()` and `.predict()`. This is why you can pass a plain Python object and sklearn will use it happily, as long as the interface matches.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score

# This class has NO sklearn inheritance whatsoever
class MedianClassifier:
    """Predicts the majority class. Knows nothing about sklearn."""

    def fit(self, X, y):
        self._majority = y.mode()[0]
        return self

    def predict(self, X):
        return np.full(len(X), self._majority)

    def score(self, X, y):
        return (self.predict(X) == y).mean()


# sklearn doesn't care about the inheritance chain — it only cares about the interface
np.random.seed(42)
X = pd.DataFrame({"tenure": np.random.randint(1, 60, 200),
                  "charges": np.random.uniform(20, 100, 200)})
y = pd.Series((np.random.rand(200) > 0.7).astype(int))

clf = MedianClassifier()
scores = cross_val_score(clf, X, y, cv=5)
print(f"Cross-val scores: {scores.round(3)}")
print(f"Mean: {scores.mean():.3f}")
print()
print(f"Is MedianClassifier a subclass of anything sklearn? No.")
print(f"Does sklearn care? Also no.")

---
## 5. The Tension: Type Hints Meet Duck Typing <a id="5"></a>

Here's where things get interesting. In notebook 04 we wrote:

```python
def evaluate_preprocessors(
    preprocessors: list[BasePreprocessor],
    ...
) -> pd.DataFrame:
```

That type hint says: *"this function accepts a list of `BasePreprocessor` instances."*

But duck typing says: *"this function works with anything that has `.fit_transform()` and `.transform()`."*

These are two different kinds of compatibility:

| Kind | What it means | Checked by |
|---|---|---|
| **Hierarchical** | `StandardScaler` inherits from `BasePreprocessor` | `isinstance()`, mypy, the ABC |
| **Structural** | Object has `.fit()` and `.transform()` methods | Python at runtime (duck typing) |

The problem: if you pass a `MedianClassifier` (which has `.fit()` but not `.transform()`) to `evaluate_preprocessors`, mypy will complain — it's not a `BasePreprocessor`. But if you pass sklearn's own `StandardScaler`, mypy will *also* complain, even though it works perfectly at runtime, because sklearn's `StandardScaler` doesn't inherit from *your* `BasePreprocessor`.

**Forcing hierarchical inheritance to express structural compatibility is the wrong tool.** That's where Protocols come in.

In [ ]:
from abc import ABC, abstractmethod

class BasePreprocessor(ABC):
    @abstractmethod
    def fit(self, X: pd.DataFrame) -> "BasePreprocessor": ...
    @abstractmethod
    def transform(self, X: pd.DataFrame) -> pd.DataFrame: ...
    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)


def evaluate_preprocessors(
    preprocessors: list[BasePreprocessor],   # <-- hierarchical type hint
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
) -> list[pd.DataFrame]:
    results = []
    for p in preprocessors:
        X_tr = p.fit_transform(X_train)
        X_te = p.transform(X_test)
        results.append(X_te)
    return results


# sklearn's StandardScaler has fit_transform() and transform() — it works at runtime
from sklearn.preprocessing import StandardScaler as SklearnScaler

X_train = pd.DataFrame({"a": [1.0, 2.0, 3.0, 4.0], "b": [10.0, 20.0, 30.0, 40.0]})
X_test  = pd.DataFrame({"a": [5.0, 6.0], "b": [50.0, 60.0]})

# This runs fine at runtime...
results = evaluate_preprocessors([SklearnScaler()], X_train, X_test)
print("Runtime: works fine with sklearn's scaler")
print(results[0].round(2))
print()
# ...but mypy would flag it:
# error: Argument 1 has incompatible type "list[StandardScaler]";
#        expected "list[BasePreprocessor]"
# Because SklearnScaler doesn't inherit from YOUR BasePreprocessor.
print("mypy would reject this even though it works perfectly.")
print("The type hint is too narrow — it describes hierarchy, not structure.")

---
## 6. Protocols: Structural Typing Without Inheritance <a id="6"></a>

`Protocol` (from `typing`) lets you define a type based on **what methods an object has**, not **what it inherits from**. This is called **structural subtyping** — or, formally, *duck typing that mypy can check*.

Any class that has the right methods satisfies the Protocol, regardless of its inheritance chain. No `isinstance()` check, no ABC, no `register()` call — just the methods.

This is the correct tool when you want to express: *"I accept anything that looks like a preprocessor."*

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Preprocessor(Protocol):
    """Structural type: anything with fit(), transform(), and fit_transform()."""

    def fit(self, X: pd.DataFrame) -> "Preprocessor": ...
    def transform(self, X: pd.DataFrame) -> pd.DataFrame: ...
    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame: ...


# Now the function accepts ANYTHING with the right methods
def evaluate_preprocessors_v2(
    preprocessors: list[Preprocessor],   # structural — not hierarchical
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
) -> list[pd.DataFrame]:
    results = []
    for p in preprocessors:
        X_tr = p.fit_transform(X_train)
        results.append(p.transform(X_test))
    return results


# Our own scaler
class MyScaler:
    def fit(self, X):
        self._mean = X.mean(); return self
    def transform(self, X):
        return X - self._mean
    def fit_transform(self, X):
        return self.fit(X).transform(X)

# sklearn's scaler
from sklearn.preprocessing import StandardScaler as SklearnScaler

# Both satisfy the Protocol — mypy is happy, runtime is happy
for scaler in [MyScaler(), SklearnScaler()]:
    print(f"{type(scaler).__name__:20s}  satisfies Preprocessor: {isinstance(scaler, Preprocessor)}")

results = evaluate_preprocessors_v2([MyScaler(), SklearnScaler()], X_train, X_test)
print()
print("Both work — no inheritance required, mypy accepts both.")

In [ ]:
# The three approaches side by side

print("=== Approach 1: No type hint (pure duck typing) ===")
print("  def fn(preprocessors: list):  ...")
print("  + Works with anything")
print("  - mypy can't help you, IDE has no autocomplete")
print()

print("=== Approach 2: ABC type hint (hierarchical) ===")
print("  def fn(preprocessors: list[BasePreprocessor]):  ...")
print("  + mypy checks inheritance chain")
print("  - Rejects valid objects that don't inherit from YOUR base class")
print("  - Forces sklearn, third-party, and legacy code to subclass you")
print()

print("=== Approach 3: Protocol (structural) ===")
print("  def fn(preprocessors: list[Preprocessor]):  ...")
print("  + mypy checks method signatures, not inheritance")
print("  + Works with sklearn, any third-party library, legacy code")
print("  + No changes required to existing classes")
print()
print("Rule of thumb:")
print("  Use ABCs when you OWN the class hierarchy and want to enforce implementation.")
print("  Use Protocols when you want to describe an interface others already satisfy.")

---
## 7. Exercise: A Saveable Model <a id="7"></a>

Create a `Fittable` Protocol with methods `fit(X, y)` and `predict(X)`.

Then implement `PersistableRandomForest` that:
1. Inherits from `PersistenceMixin` and `FittedCheckMixin` (from section 2)
2. Wraps sklearn's `RandomForestClassifier`
3. Calls `_require_fitted()` inside `predict()`
4. Can be saved and reloaded with `save()` / `load()`

Confirm with `isinstance(model, Fittable)` that it satisfies the Protocol — without inheriting from it.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import tempfile, os

# Define the Protocol
class Fittable(Protocol):
    def fit(self, X: pd.DataFrame, y: pd.Series) -> "Fittable": ...
    def predict(self, X: pd.DataFrame) -> np.ndarray: ...


class PersistableRandomForest(PersistenceMixin, FittedCheckMixin):
    pass   # TODO


# Test
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(0)
X_all = pd.DataFrame({"a": np.random.randn(100), "b": np.random.randn(100)})
y_all = pd.Series((X_all["a"] > 0).astype(int))
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=0)

model = PersistableRandomForest()

try:
    model.predict(X_te)
except RuntimeError as e:
    print(f"RuntimeError: {e}")

model.fit(X_tr, y_tr)
with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as f:
    tmp = f.name
try:
    model.save(tmp)
    reloaded = PersistableRandomForest.load(tmp)
    acc = accuracy_score(y_te, reloaded.predict(X_te))
    print(f"Accuracy after reload: {acc:.3f}")
    print(f"Satisfies Fittable Protocol: {isinstance(reloaded, Fittable)}")
finally:
    os.unlink(tmp)

> <details>
> <summary>Solution</summary>
>
> ```python
> class PersistableRandomForest(PersistenceMixin, FittedCheckMixin):
>     def __init__(self, n_estimators: int = 100, random_state: int = 42):
>         self._fitted = False
>         self._model = RandomForestClassifier(
>             n_estimators=n_estimators, random_state=random_state
>         )
>
>     def fit(self, X, y):
>         self._model.fit(X, y)
>         self._fitted = True
>         return self
>
>     def predict(self, X):
>         self._require_fitted()
>         return self._model.predict(X)
> ```
> Note: `PersistableRandomForest` satisfies `Fittable` purely structurally —
> it has `fit()` and `predict()` with the right signatures. No inheritance needed.
> </details>